# Reviewer 3 Comment 9: Original-code ligand feature path (executed)

This notebook loads and executes **unmodified functional_groups.py from the uploaded reconstructed project ZIP**, checks the actual graph-building default, and tests the reviewer's molecules. To rerun, provide `Fg-PVgraphDTA-main.zip` next to the notebook (or set `FG_SOURCE_ZIP`); RDKit and NumPy are required. **No model weights, training or architecture are changed.** The archive is a RECONSTRUCTION, not the original final trained-run graph artifacts; historical training provenance remains unverified.


In [1]:
from pathlib import Path
from zipfile import ZipFile
import os, sys, hashlib
import numpy as np
from rdkit import Chem, rdBase
source_zip=Path(os.getenv("FG_SOURCE_ZIP","/mnt/data/Fg-PVgraphDTA-main.zip"))
if not source_zip.exists(): source_zip=Path("Fg-PVgraphDTA-main.zip")
if not source_zip.exists(): raise FileNotFoundError("Place the ORIGINAL supplied Fg-PVgraphDTA-main.zip beside this notebook (or set FG_SOURCE_ZIP).")
with ZipFile(source_zip) as z:
    prefix="Fg-PVgraphDTA-main/"
    fs={p:z.read(prefix+p).decode("utf-8") for p in [
       "src/fgpvdta/preprocessing/functional_groups.py",
       "src/fgpvdta/preprocessing/ligand_graphs.py",
       "src/fgpvdta/preprocessing/feature_sets.py",
       "src/fgpvdta/preprocessing/pipeline.py",
       "docs/HISTORICAL_DIFFERENCES.md"]}
fg_source=fs["src/fgpvdta/preprocessing/functional_groups.py"]
module=type(sys)("original_functional_groups")
exec(compile(fg_source,"functional_groups.py","exec"),module.__dict__)
print("RDKit:",rdBase.rdkitVersion)
print("functional_groups.py SHA256:",hashlib.sha256(fg_source.encode()).hexdigest())
print("SOURCE:",source_zip)


RDKit: 2025.09.4
functional_groups.py SHA256: 9c7bc7a81ccad3cb165fbc1a4a5feb97a1651bb8f378202d4ba1085b145d8406
SOURCE: /mnt/data/Fg-PVgraphDTA-main.zip


In [2]:
source=fs["src/fgpvdta/preprocessing/functional_groups.py"]
lig=fs["src/fgpvdta/preprocessing/ligand_graphs.py"]
feat=fs["src/fgpvdta/preprocessing/feature_sets.py"]
pipe=fs["src/fgpvdta/preprocessing/pipeline.py"]
assert 'if overlap_policy == "notebook":' in source
assert 'fg_overlap_policy: Literal["notebook", "clean"] = "notebook"' in lig
assert 'atom_functional_group_matrix(mol, fg_overlap_policy)' in lig
assert 'base = atom_functional_group_matrix(mol)' in feat
assert 'fg = ligand_features(mol, count)' in pipe
assert "cleanup_group_names(" in source
print("PASS: reconstructed main pipeline -> ligand_features -> atom_functional_group_matrix(default='notebook', RAW)")
print("PASS: ligand graph builder also defaults to 'notebook' (RAW)")
print("PASS: detect_functional_groups metadata calls cleanup_group_names, BUT per-atom RAW input bypasses cleanup")
print("SOURCE DOCUMENTATION:",[x for x in fs["docs/HISTORICAL_DIFFERENCES.md"].splitlines() if "Molecule-level metadata applies cleanup" in x][0])


PASS: reconstructed main pipeline -> ligand_features -> atom_functional_group_matrix(default='notebook', RAW)
PASS: ligand graph builder also defaults to 'notebook' (RAW)
PASS: detect_functional_groups metadata calls cleanup_group_names, BUT per-atom RAW input bypasses cleanup
SOURCE DOCUMENTATION: Molecule-level metadata applies cleanup (Ester>Ether, CarboxylicAcid>Alcohol, Phenol>Alcohol). Historical per-atom ligand FG bits do not call that cleanup. `fg_overlap_policy=notebook` reproduces this behavior.


In [3]:
cols={n:i for i,n in enumerate(module.FG_NAMES)}
def active(m): return [n for n,i in cols.items() if m[:,i].any()]
def shared(m,a,b):return np.flatnonzero((m[:,cols[a]]>0)&(m[:,cols[b]]>0)).tolist()
for name,smi in [('Acetic acid','CC(=O)O'),('Aspirin','CC(=O)Oc1ccccc1C(=O)O'),('Ethyl acetate','CC(=O)OCC'),('Phenol','Oc1ccccc1'),('Lactic acid (separate true alcohol)','CC(O)C(=O)O'),('2-methoxyethyl acetate (separate true ether)','CC(=O)OCCOC')]:
    mol=Chem.MolFromSmiles(smi)
    raw=module.atom_functional_group_matrix(mol)
    clean=module.atom_functional_group_matrix(mol,"clean")
    meta=module.detect_functional_groups(mol)
    assert not shared(clean,"CarboxylicAcid","Alcohol")
    assert not shared(clean,"Ester","Ether")
    print(name, "| metadata:",meta,"| raw:",active(raw),"| optional clean:",active(clean))
    if name=="Acetic acid":
        assert set(active(raw))=={"CarboxylicAcid","Alcohol"}
        assert active(clean)==["CarboxylicAcid"]
    if name=="Aspirin":
        assert {"CarboxylicAcid","Ester","Alcohol","Ether","Aryl"}<=set(active(raw))
    if name=="Ethyl acetate":assert shared(raw,"Ester","Ether")
    if name.startswith("Lactic"):assert "Alcohol" in active(clean)
    if name.startswith("2-methoxyethyl"):assert "Ether" in active(clean)
print("PASS: reviewer false positives occur in RAW atom-level input; CLEAN mode removes same-atom overlap without removing independent OH/ether in these controls.")


Acetic acid | metadata: ['CarboxylicAcid'] | raw: ['CarboxylicAcid', 'Alcohol'] | optional clean: ['CarboxylicAcid']
Aspirin | metadata: ['Aryl', 'CarboxylicAcid', 'Ester'] | raw: ['CarboxylicAcid', 'Ester', 'Alcohol', 'Ether', 'Aryl'] | optional clean: ['CarboxylicAcid', 'Ester', 'Aryl']
Ethyl acetate | metadata: ['Ester'] | raw: ['Ester', 'Ether'] | optional clean: ['Ester']
Phenol | metadata: ['Aryl', 'Phenol'] | raw: ['Phenol', 'Aryl'] | optional clean: ['Phenol', 'Aryl']
Lactic acid (separate true alcohol) | metadata: ['CarboxylicAcid'] | raw: ['CarboxylicAcid', 'Alcohol'] | optional clean: ['CarboxylicAcid', 'Alcohol']
2-methoxyethyl acetate (separate true ether) | metadata: ['Ester'] | raw: ['Ester', 'Ether'] | optional clean: ['Ester', 'Ether']
PASS: reviewer false positives occur in RAW atom-level input; CLEAN mode removes same-atom overlap without removing independent OH/ether in these controls.


## Conclusion

The reconstructed default **per-atom input** is `notebook`/raw and DOES NOT invoke molecule-level cleanup; the separate metadata and residue-vector cleanup is not proof of clean ligand-atom model inputs. Acetic acid, aspirin and ethyl acetate reproduce the reviewer's overlap. The optional `clean` policy is NOT the historical default. Source audit alone cannot prove which features the final historical checkpoint consumed; inspect the actual original ligand-graph tensors / original training notebooks for that. A BEFORE/AFTER performance ablation requires model retraining; no performance numbers are claimed here. For a fuller executed notebook including dataset-level overlap counts, see the separately generated local artifact.
